Importance Sampling
---
Using sampling to approximate a distribution

$$E[f(x)] = \int f(x)p(x) dx \approx \frac{1}{n}\sum_{i} f(x_i)$$
where $ x \sim p(x)$

$$E[f(x)] = \int f(x)p(x) dx = \int f(x)\frac{p(x)}{q(x)}q(x) dx \approx \frac{1}{n} \sum_{i} f(x_i)\frac{p(x_i)}{q(x_i)}$$

where $ x \sim q(x)$

Idea of importance sampling: draw the sample from a proposal distribution and re-weight the integral using importance weights so that the correct distribution is targeted

$$Var(X) = E[X^2] - E[X]^2$$

**Reference**

- [1](https://www.youtube.com/watch?v=3Mw6ivkDVZc)
- [2](https://astrostatistics.psu.edu/su14/lectures/cisewski_is.pdf)

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Define f(x)
def f_x(x):
    return 1/(1 + np.exp(-x))

In [ ]:
plt.figure(figsize=[6, 4])
x = np.linspace(0, 4, 50)  # x ranges from 0 to 4
y = [f_x(i) for i in x] # outputs

plt.plot(x, y, label="$f(x)$")

plt.xlabel("x", size=18)
plt.ylabel("y", size=18)
plt.legend(prop={"size": 14})

## Sampling

In [ ]:
# For simlicity, we use normal distribution for p(x) and q(x)
def distribution(mu=0, sigma=1):
    # return probability given a value
    distribution = stats.norm(mu, sigma)
    return distribution

In [ ]:
# pre-setting p and q
n = 1000

mu_target = 3.5     # mean for p(x)
sigma_target = 1    # std for p(x)
mu_appro = 3        # mean for q(x)
sigma_appro = 1     # std for q(x)

p_x = distribution(mu_target, sigma_target) # target distribution
q_x = distribution(mu_appro, sigma_appro)

In [ ]:
# Plot p and q
plt.figure(figsize=[10, 4])

sns.distplot([np.random.normal(mu_target, sigma_target) for _ in range(3000)], label="distribution $p(x)$") # plot p
sns.distplot([np.random.normal(mu_appro, sigma_appro) for _ in range(3000)], label="distribution $q(x)$") # plot q

plt.title("Distributions", size=16)
plt.legend()

In [ ]:
# Compute the true value sample from p(x)
s = 0
for i in range(n):
    # draw a sample
    x_i = np.random.normal(mu_target, sigma_target)
    s += f_x(x_i)
print("simulate value", s/n) # Compute the expectation value

In [ ]:
# calculate value sampling from distribution q (hint: use importantce sampling equation)

# Print the expectation value (mean and variance)

## Different $q(x)$

In [ ]:
# pre-setting
n = 5000 # we need more samples to approximate the value, when distribution is dissimilar

mu_target = 3.5     # mean for p(x)
sigma_target = 1    # std for p(x)
mu_appro = 1        # mean for q(x)
sigma_appro = 1     # std for q(x)

# Rejection Sampling

In [ ]:
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Define target distribution p using a simple Gaussian mixture model.
def p(x):
    return norm.pdf(x, loc=30, scale=10) + norm.pdf(x, loc=80, scale=20)

In [ ]:
# Define proposal distribution q
def q(x):
    return norm.pdf(x, loc=50, scale=30)

In [ ]:
# Plot p and q
x = np.arange(-50, 151)
fig, ax = plt.subplots()
ax.plot(x, p(x), label=r"$p(x)$")
ax.plot(x, q(x), label=r"$q(x)$")
plt.legend()
plt.show()

In [ ]:
# Define constant k
k = max(p(x) / q(x)) # Try different values for k

In [ ]:
# Plot the scaled q distribution
fig, ax = plt.subplots()
ax.plot(x, p(x), label=r"$p(x)$")
ax.plot(x, k * q(x), label=r"$k \cdot q(x)$")
plt.legend()
plt.show()

In [ ]:
# Define a function of rejection sampling for approximation (hint: while loop or numpy vectorization)


In [ ]:
# Calculate the acceptance ratio
n_samples = 10000 # total number of samples used
accept_ratios = # number of accepted samples/ total number of samples
print(accept_ratios)

In [ ]:
# Plot approximation of the target distribution
sns.distplot()
plt.show()

# MCMC

In [ ]:
# NOTE: p(x) here is an unnormalized mixture (integrates to 2).
# For plotting against density=True histograms, use p(x)/2 as the normalized target pdf.
def p_pdf(x):
    return p(x) / 2.0

In [ ]:
import pymc as pm
import numpy as np

# log target for MH (unnormalized is fine)
def logp_target(x):
    logp1 = pm.logp(pm.Normal.dist(mu=30.0, sigma=10.0), x)
    logp2 = pm.logp(pm.Normal.dist(mu=80.0, sigma=20.0), x)
    return pm.math.logaddexp(logp1, logp2)

draws = 20000
tune  = 2000
burn_in = tune

with pm.Model() as model:
    x_mcmc = pm.Flat("x")                 # no prior; target specified by Potential
    pm.Potential("target", logp_target(x_mcmc))

    step = pm.Metropolis()
    idata = pm.sample(draws=draws, tune=tune, step=step,
                      chains=1, cores=1, random_seed=42, progressbar=True)

mh_samples = idata.posterior["x"].values.reshape(-1)
mh_post = mh_samples[burn_in:]

print("MH post-burn-in samples:", len(mh_post))

In [ ]:
#acceptance rate


In [ ]:
#MH Trace Plot

In [ ]:
#MH Samples vs Target

In [ ]:
# Importance sampling using the SAME proposal as your rejection sampling code
N = 10000
x_prop = np.random.normal(50, 30, size=N)  # same proposal q sampler as in RS section

# importance weights: w ∝ p(x)/q(x)
w = p(x_prop) / (q(x_prop) + 1e-300)
w = w / w.sum()

# resample to get unweighted samples for histogram comparison
idx = np.random.choice(np.arange(N), size=N, replace=True, p=w)
is_samples = x_prop[idx]

# (optional) ESS for a quick diagnostic
ess = 1.0 / np.sum(w**2)
print("IS ESS =", ess, "out of", N)

In [ ]:
#Task 7

xs = np.linspace(-50, 150, 800)

rs_samples = sample(10000)  # your existing rejection sampler
# mh_post is your MH post-burn-in samples from the PyMC section

plt.figure(figsize=(9,4))
plt.plot(xs, p(xs)/2.0, linewidth=2.5, label="target pdf (normalized)")

plt.hist(is_samples, bins=80, density=True, alpha=0.30, label="Importance sampling (resampled)")
plt.hist(rs_samples, bins=80, density=True, alpha=0.30, label="Rejection sampling (accepted)")
plt.hist(mh_post,    bins=80, density=True, alpha=0.30, label="MH (post burn-in)")

plt.title("IS vs RS vs MH")
plt.xlabel("x")
plt.ylabel("density")
plt.legend()
plt.tight_layout()
plt.show()